In [ ]:
!pip install -q anthropic openai yfinance

In [ ]:
import anthropic
import yfinance as yf
from openai import OpenAI
from datetime import datetime
from zoneinfo import ZoneInfo
from google.colab import userdata

claude = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY").strip(), timeout=300, max_retries=1)
chatgpt = OpenAI(api_key=userdata.get("OPENAI_API_KEY").strip(), timeout=300, max_retries=1)
grok = OpenAI(
    api_key=userdata.get("XAI_API_KEY").strip(),
    base_url="https://api.x.ai/v1",
    timeout=300,
    max_retries=1,
)

CLAUDE_MODEL = "claude-sonnet-5-5"
OPENAI_MODEL = "gpt-6-luna"
GROK_MODEL = "grok-4.7"

# Keeps research from earlier runs so the same stock is not researched twice in one day.
try:
    RESEARCH_CACHE
except NameError:
    RESEARCH_CACHE = {}

print("Setup done.")

In [ ]:
HOW_TO_READ = """HOW TO READ THESE NUMBERS:
- Forward P/E LOWER than trailing P/E means analysts expect earnings to RISE.
- Forward P/E HIGHER than trailing P/E means analysts expect earnings to FALL, or last year had a one-time gain.
- Revenue growth is a growth rate. It does not tell you whether growth is speeding up or slowing down.
- FCF yield is free cash flow divided by market cap. Under 1% is very low. Over 5% is high.
- Net cash is cash minus debt. Negative means the company owes more than it holds.
- Analyst price targets are opinions about where the price may be in about 12 months. They are often wrong and tend to lean optimistic.
- Short interest is the share of the stock that traders have bet against. Above 10% of float is high.
- Beta above 1 means the stock usually moves more than the overall market. Below 1 means less."""

def money(x):
    if x is None:
        return "n/a"
    sign = "-" if x < 0 else ""
    x = abs(x)
    for unit, size in (("T", 1e12), ("B", 1e9), ("M", 1e6)):
        if x >= size:
            return f"{sign}${x / size:.2f}{unit}"
    return f"{sign}${x:,.2f}"

def pct(x):
    return "n/a" if x is None else f"{x * 100:.1f}%"

def num(x):
    return "n/a" if x is None else f"{x:.1f}"

def get_facts(ticker):
    info = yf.Ticker(ticker).info
    if not info or info.get("marketCap") is None:
        raise ValueError(f"No data found for {ticker}. Check the ticker and try again.")

    cap = info.get("marketCap")
    fcf = info.get("freeCashflow")
    cash = info.get("totalCash")
    debt = info.get("totalDebt")
    price = info.get("currentPrice")
    target = info.get("targetMeanPrice")
    fcf_yield = None if fcf is None or not cap else fcf / cap
    net_cash = None if cash is None or debt is None else cash - debt
    target_gap = None if not price or not target else target / price - 1
    today = datetime.now(ZoneInfo("America/Los_Angeles")).date()
    name = info.get("shortName")

    lines = [
        f"Data pulled on {today} from Yahoo Finance",
        f"Company: {name} ({ticker})",
        f"Sector: {info.get('sector')}",
        f"Industry: {info.get('industry')}",
        f"Price: {money(price)}",
        f"52-week range: {money(info.get('fiftyTwoWeekLow'))} to {money(info.get('fiftyTwoWeekHigh'))}",
        f"Market cap: {money(cap)}",
        f"Revenue (last 12 months): {money(info.get('totalRevenue'))}",
        f"Revenue growth (latest quarter vs a year ago): {pct(info.get('revenueGrowth'))}",
        f"Gross margin: {pct(info.get('grossMargins'))}",
        f"Profit margin: {pct(info.get('profitMargins'))}",
        f"Free cash flow: {money(fcf)}",
        f"FCF yield: {pct(fcf_yield)}",
        f"Cash: {money(cash)}",
        f"Debt: {money(debt)}",
        f"Net cash: {money(net_cash)}",
        f"P/E (trailing): {num(info.get('trailingPE'))}",
        f"P/E (forward): {num(info.get('forwardPE'))}",
        f"Price to sales: {num(info.get('priceToSalesTrailing12Months'))}",
        f"Analyst average price target: {money(target)} ({pct(target_gap)} vs today's price)",
        f"Analyst target range: {money(info.get('targetLowPrice'))} to {money(info.get('targetHighPrice'))}",
        f"Number of analysts: {info.get('numberOfAnalystOpinions') or 'n/a'}",
        f"Analyst consensus label: {info.get('recommendationKey') or 'n/a'}",
        f"Short interest: {pct(info.get('shortPercentOfFloat'))} of float",
        f"Beta: {num(info.get('beta'))}",
    ]
    return "\n".join(lines), name, today

print("Data step ready.")

In [ ]:
HORIZONS = {
    "short": "a short-term trade lasting days to a few weeks",
    "long": "a long-term hold lasting a year or more",
}

FOCUS = {
    "short": """
Focus on what could move the stock over the next few days and weeks:
- news from the last 14 days
- upcoming dated events: the next earnings date, product launches, court dates,
  and economic reports that matter for this sector
- analyst upgrades, downgrades and price target changes in the last 30 days
- how the sector and its closest competitors have moved recently
- unusual trading volume, and any short squeeze or momentum story
Cover the latest earnings report in at most 2 bullets.
""",
    "long": """
Focus on what drives the business over years:
- what the business does and its main segments
- the latest earnings report, guidance, and what management said
- competitive position and market share
- cash, debt and spending plans
- major deals, customers and new projects
- legal, political and regulatory issues
- the main long-term risks
Also give the next earnings date if it is known.
""",
}

def x_sentiment(ticker, name, today):
    # Asks Grok what traders on X are saying. Billed per post fetched, so it is kept small.
    try:
        r = grok.responses.create(
            model=GROK_MODEL,
            input=[{
                "role": "user",
                "content": (
                    f"Search X for posts from the last 7 days about ${ticker} ({name}). "
                    "Use at most 2 searches. Summarize what traders are saying in at most 5 bullets, "
                    "each starting with a dash: the main bullish view, the main bearish view, "
                    "any specific rumor or event being discussed, and whether the mood is mostly "
                    "positive, mostly negative or split. These are opinions. Label rumors as rumors "
                    "and never present a post as a confirmed fact. Include 2 or 3 post links as examples. "
                    "Plain text, no markdown symbols."
                ),
            }],
            tools=[{
                "type": "x_search",
                "from_date": str(today - timedelta(days=7)),
                "to_date": str(today + timedelta(days=1)),
            }],
            extra_body={"max_turns": 2},
        )
        try:
            used = (r.usage.model_dump().get("server_side_tool_usage_details") or {})
            posts = used.get("x_posts_fetched")
            if posts:
                print(f"    (X search read {posts} posts, about ${posts * 0.005:.2f})")
        except Exception:
            pass
        return r.output_text
    except Exception as e:
        return f"- X search was not available for this run ({type(e).__name__})."

def research(ticker, name, today, horizon):
    key = (ticker, str(today), horizon)
    if key in RESEARCH_CACHE:
        print("    (using research saved from earlier today, no Grok search cost)")
        return RESEARCH_CACHE[key]

    prompt = (
        f"Today is {today}. Research the company {name} ({ticker}) using web search.\n"
        f"The reader is considering {HORIZONS[horizon]}.\n"
        + FOCUS[horizon]
        + """
Report facts only, with no opinion on whether to buy or sell.
Prefer the company's own filings and earnings releases for numbers.
Include trailing free cash flow as the company reports it.
Hard limit: 14 bullets total, at most 2 sentences each. Start each bullet with a dash.
Every bullet needs its date and one source URL. Include good news and bad news.
Skip anything you cannot source. Plain text, no markdown symbols.
"""
    )
    r = grok.responses.create(
        model=GROK_MODEL,
        input=[{"role": "user", "content": prompt}],
        tools=[{"type": "web_search"}],
    )
    findings = r.output_text

    if horizon == "short":
        findings += (
            "\n\nWHAT TRADERS ARE SAYING ON X (opinions, not facts):\n"
            + x_sentiment(ticker, name, today)
        )

    RESEARCH_CACHE[key] = findings
    return findings

print("Research step ready.")

In [ ]:
BULL_MODEL = "claude"   # change to "chatgpt" to swap sides

RULES = """
Use ONLY the DATA block and the RESEARCH block for facts and figures.
Never invent a number or a fact. If something you need is in neither block, say "not in the research".
If a number in DATA conflicts with a number from a company filing in RESEARCH, the filing wins.
Use the filing number and say that the two sources conflict.
Posts from X are opinions and rumors. You may describe the mood, but never use a post as proof of a fact.
Read the HOW TO READ sections before interpreting any number.
Use only the chart levels given in PRICE CHART DATA. Never invent a price level.
Argue about the stock itself. Do not build your case on what is missing from the data.
Write in plain text with no markdown symbols.
"""

def ask_claude(system, prompt):
    r = claude.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=4000,
        system=system + "\n" + RULES,
        messages=[{"role": "user", "content": prompt}],
    )
    # Keep only the answer text. Skip any reasoning blocks the model sends back.
    return "".join(block.text for block in r.content if block.type == "text").strip()

def ask_chatgpt(system, prompt):
    r = chatgpt.responses.create(
        model=OPENAI_MODEL,
        instructions=system + "\n" + RULES,
        input=prompt,
    )
    return r.output_text

ASK = {"claude": ask_claude, "chatgpt": ask_chatgpt}
NAMES = {"claude": "Claude", "chatgpt": "ChatGPT"}

def run_debate(data, horizon):
    bull_key = BULL_MODEL
    bear_key = "chatgpt" if bull_key == "claude" else "claude"
    ask_bull, ask_bear = ASK[bull_key], ASK[bear_key]
    frame = (
        f"The reader is considering {HORIZONS[horizon]}. "
        "Argue for that time frame and use the evidence that matters most for it."
    )

    # Openings: each side writes without seeing the other.
    bull = ask_bull(
        "You are a bull analyst. Make the strongest honest case FOR buying this stock, "
        "using the numbers, the chart data and the research. Max 230 words. " + frame,
        data,
    )
    bear = ask_bear(
        "You are a bear analyst. Make the strongest honest case AGAINST buying this stock, "
        "using the numbers, the chart data and the research. Max 230 words. " + frame,
        data,
    )

    # Replies: each side answers the other's opening. Neither sees the other's reply.
    bull_reply = ask_bull(
        "You are the bull analyst. Below is your opening case and the bear's opening case. "
        "Attack the weakest points in the bear case and call out any number or fact it got wrong. "
        "Concede only the points where the bear is actually right. Max 170 words. " + frame,
        f"{data}\n\nYour bull case:\n{bull}\n\nBear case:\n{bear}",
    )
    bear_reply = ask_bear(
        "You are the bear analyst. Below is your opening case and the bull's opening case. "
        "Attack the weakest points in the bull case and call out any number or fact it got wrong. "
        "Concede only the points where the bull is actually right. Max 170 words. " + frame,
        f"{data}\n\nYour bear case:\n{bear}\n\nBull case:\n{bull}",
    )

    return {
        "bull": bull, "bear": bear,
        "bull_reply": bull_reply, "bear_reply": bear_reply,
        "bull_name": NAMES[bull_key], "bear_name": NAMES[bear_key],
    }

print(f"Debate step ready. Bull = {NAMES[BULL_MODEL]}.")

In [ ]:
JUDGE_INSTRUCTIONS = """
You are a neutral judge of a stock debate. You have the full DATA and RESEARCH
below, plus the debate. Judge the EVIDENCE, not the debating. Check every claim
against the DATA and RESEARCH yourself. A point counts only if the evidence
supports it, no matter how well or badly it was argued. If strong evidence
exists for a side that the debater failed to use, include it anyway. Do not
reward a side for catching the other's mistakes. Posts from X are opinions,
never proof of a fact.

The reader is considering {HORIZON}. Weigh the evidence for that time frame.
For a short-term trade, near-term events, trader mood and chart levels matter
most. For a long-term hold, business quality, cash and valuation matter most.

HOW TO WRITE:
- Write complete, clear sentences. Being clear matters more than being short.
- Use the real name of every financial term. The first time a term appears,
  explain it in a few words in brackets, for example "RSI (a 0 to 100 gauge of
  how fast the price has moved)" or "free cash flow (cash left after spending
  on equipment)". Never invent a nickname for a standard term.
- One idea per sentence. Do not cram several facts into one sentence with commas.

Write a brief for a regular investor who wants enough evidence to make their
own decision. Use exactly these nine headings, written exactly as shown with
nothing added, each on its own line, with a blank line between sections:

WHERE THE EVIDENCE LEANS
THE DECISION
BEST BULL POINTS
BEST BEAR POINTS
WHERE THEY DISAGREE
WHAT THE CHART SHOWS
DATA WARNINGS
CLAIMS THAT FAILED
CHECK THIS YOURSELF

Under WHERE THE EVIDENCE LEANS, write exactly one line. It must start with
"Leans bullish:" or "Leans bearish:" or "Mixed:" and then give the single
biggest reason for this time frame. Base it only on the weight of the evidence,
never on who debated better. Choose "Mixed:" unless the evidence clearly favors
one side.

Under THE DECISION, write exactly these seven lines, each starting with a dash
and the label shown, each one or two clear sentences:
- The bet: what someone buying at today's price for this time frame is betting will happen.
- You're wrong if: the specific things that would break that bet.
- What the price already assumes: how much good news is already in the price, using the valuation numbers and the analyst price target versus today's price. Remind the reader that analyst targets are opinions.
- Fits you if: describe the person this suits, in terms of how long they can hold, how much of a drop they can sit through, and how closely they can watch it.
- Doesn't fit you if: describe the person this does not suit, in the same terms.
- Case gets stronger if: one or two concrete, checkable signals, using real numbers, price levels or dates from the data.
- Case gets weaker if: one or two concrete, checkable signals, using real numbers, price levels or dates from the data.
Do not tell the reader to buy or sell. Lay out the decision so they can make it.

Under BEST BULL POINTS give the 3 best-supported points for owning the stock.
Under BEST BEAR POINTS give the 3 best-supported points against.
Write EACH of those six points as exactly four lines in this form:
- The point in one plain sentence.
Evidence: the specific numbers or facts behind it, and where they come from (company filing, earnings call, named news source).
Why it matters: what this means for someone deciding whether to own the stock for this time frame.
Other side: the strongest counter-argument from the evidence, or "No strong counter in the evidence."
Each of those lines is one or two clear sentences.

Under WHERE THEY DISAGREE, give 1 or 2 questions the evidence cannot settle,
each on its own line starting with a dash, and say what would settle each one.

Under WHAT THE CHART SHOWS, write for a beginner, using ONLY the numbers in the
PRICE CHART DATA section. Never invent a price level. Give exactly 5 points,
each on its own line starting with a dash:
1. The trend, and what it means in plain words.
2. The nearest support level, with a few words explaining what support is.
3. The nearest resistance level, with a few words explaining what resistance is.
4. Start with "Bull read:" and say what chart traders would see as a good sign from here.
5. Start with "Bear read:" and say what chart traders would see as a warning sign from here.
If PRICE CHART DATA says no chart data is available, write a dash and "No chart data."
Chart levels describe past behavior. Never present them as predictions.

Under DATA WARNINGS, list numbers where sources conflicted and which to trust,
each on its own line starting with a dash, or write a dash and the word None.
Under CLAIMS THAT FAILED, list up to 3 claims from either side that the
evidence contradicts or does not support, and say which side made each.
Under CHECK THIS YOURSELF, give 2 or 3 things to check.

Hard limit: 950 words total. Plain text, no markdown symbols.
"""

def judge(data, d, horizon):
    instructions = JUDGE_INSTRUCTIONS.replace("{HORIZON}", HORIZONS[horizon])
    debate_text = (
        data
        + "\n\nBULL CASE:\n" + d["bull"]
        + "\n\nBEAR CASE:\n" + d["bear"]
        + "\n\nBULL REPLY:\n" + d["bull_reply"]
        + "\n\nBEAR REPLY:\n" + d["bear_reply"]
    )

    # Stream the answer so the connection stays alive and progress is visible.
    stream = grok.with_options(timeout=240, max_retries=0).responses.create(
        model=GROK_MODEL,
        input=[{"role": "user", "content": instructions + "\n\n" + debate_text}],
        stream=True,
    )
    parts, count = [], 0
    print("    judge is thinking and writing ", end="", flush=True)
    for event in stream:
        if getattr(event, "type", "") == "response.output_text.delta":
            parts.append(event.delta)
            count += 1
            if count % 40 == 0:
                print(".", end="", flush=True)
    print()

    text = "".join(parts).strip()
    if not text:
        raise RuntimeError("The judge returned no text. Run the cell again.")
    return text

print("Judge step ready.")

In [ ]:
import json
import requests
from datetime import timedelta, timezone

POLY_KEY = userdata.get("POLYGON_API_KEY").strip()

CHART_GUIDE = """HOW TO READ THE CHART NUMBERS:
- Support is a price where buyers stepped in before and the stock bounced. More touches means a stronger level.
- Resistance is a price where sellers stepped in before and the stock turned down.
- When price falls below a support level, that level often acts as resistance afterward.
- The 50-day and 200-day averages are the average closing price over that many days. Price above both is a sign of an uptrend.
- RSI runs from 0 to 100 and measures how fast the price has moved. Above 70 is often called overbought. Below 30 is often called oversold.
- Volume above its 20-day average means more trading than usual, which makes a move more meaningful.
- All of these describe past behavior. They are not predictions."""

def poly_candles(ticker, mult, span, start, end):
    url = f"https://api.polygon.io/v2/aggs/ticker/{ticker}/range/{mult}/{span}/{start}/{end}"
    r = requests.get(
        url,
        params={"adjusted": "true", "sort": "asc", "limit": 50000},
        headers={"Authorization": f"Bearer {POLY_KEY}"},
        timeout=30,
    )
    if r.status_code != 200:
        raise ValueError(f"Polygon returned HTTP {r.status_code} for {ticker} {mult}-{span} candles.")
    rows = r.json().get("results") or []
    return [
        {"t": x["t"] // 1000, "o": x["o"], "h": x["h"], "l": x["l"], "c": x["c"], "v": x["v"]}
        for x in rows
    ]

def candle_date(c):
    return datetime.fromtimestamp(c["t"], timezone.utc).date()

def add_recent_days(ticker, daily):
    # The free Polygon plan runs a day behind, so add any newer days from Yahoo.
    try:
        recent = yf.Ticker(ticker).history(period="5d", interval="1d", auto_adjust=True)
    except Exception:
        return daily, False
    last = candle_date(daily[-1])
    added = False
    for idx, row in recent.iterrows():
        d = idx.date()
        if d > last and row["Close"] == row["Close"]:   # skips empty rows
            stamp = int(datetime(d.year, d.month, d.day, 12, tzinfo=timezone.utc).timestamp())
            daily.append({
                "t": stamp, "o": float(row["Open"]), "h": float(row["High"]),
                "l": float(row["Low"]), "c": float(row["Close"]), "v": float(row["Volume"]),
            })
            added = True
    return daily, added

def sma(values, n):
    return sum(values[-n:]) / n if len(values) >= n else None

def sma_series(values, n):
    out = []
    for i in range(len(values)):
        out.append(sum(values[i - n + 1:i + 1]) / n if i >= n - 1 else None)
    return out

def rsi(closes, n=14):
    if len(closes) < n + 1:
        return None
    gains = losses = 0.0
    for i in range(1, n + 1):
        ch = closes[i] - closes[i - 1]
        gains += max(ch, 0)
        losses += max(-ch, 0)
    avg_gain, avg_loss = gains / n, losses / n
    for i in range(n + 1, len(closes)):
        ch = closes[i] - closes[i - 1]
        avg_gain = (avg_gain * (n - 1) + max(ch, 0)) / n
        avg_loss = (avg_loss * (n - 1) + max(-ch, 0)) / n
    if avg_loss == 0:
        return 100.0
    return 100 - 100 / (1 + avg_gain / avg_loss)

def find_levels(daily, price, window=5, tol=0.015):
    # A pivot is a day whose high (or low) is the most extreme within 5 days on each side.
    pivots = []
    for i in range(window, len(daily) - window):
        seg = daily[i - window:i + window + 1]
        if daily[i]["h"] == max(x["h"] for x in seg):
            pivots.append(daily[i]["h"])
        if daily[i]["l"] == min(x["l"] for x in seg):
            pivots.append(daily[i]["l"])
    pivots.sort()

    # Pivots within 1.5% of each other count as the same level.
    clusters = []
    for p in pivots:
        if clusters and abs(p - clusters[-1][0]) / clusters[-1][0] <= tol:
            clusters[-1].append(p)
        else:
            clusters.append([p])
    levels = [(sum(c) / len(c), len(c)) for c in clusters if len(c) >= 2]

    supports = sorted([l for l in levels if l[0] < price], key=lambda l: -l[0])[:2]
    resistances = sorted([l for l in levels if l[0] > price], key=lambda l: l[0])[:2]
    return supports, resistances

def get_chart(ticker, today):
    daily = poly_candles(ticker, 1, "day", today - timedelta(days=730), today)
    if len(daily) < 60:
        raise ValueError(f"Not enough price history for {ticker}.")
    daily, added = add_recent_days(ticker, daily)

    closes = [c["c"] for c in daily]
    year = daily[-252:]
    price = closes[-1]
    last_day = candle_date(daily[-1])
    source = "Yahoo Finance" if added else "Polygon"
    stale = "" if last_day >= today - timedelta(days=1) else " WARNING: this price may be out of date."

    s50, s200 = sma(closes, 50), sma(closes, 200)
    r = rsi(closes)
    vol20 = sum(c["v"] for c in daily[-21:-1]) / 20
    last_vol = daily[-1]["v"]
    supports, resistances = find_levels(year, price)

    if s50 and s200 and price > s50 > s200:
        trend = "Uptrend: price is above the 50-day average, and the 50-day is above the 200-day."
    elif s50 and s200 and price < s50 < s200:
        trend = "Downtrend: price is below the 50-day average, and the 50-day is below the 200-day."
    elif s50 and s200 and s50 > s200 and price < s50:
        trend = "Pullback: the 50-day average is above the 200-day, but price has dropped below the 50-day."
    else:
        trend = "Mixed: price and the two averages do not line up in one direction."

    def change(days):
        return None if len(closes) <= days else closes[-1] / closes[-1 - days] - 1

    def fmt(levels):
        if not levels:
            return "none found in the last year"
        return "; ".join(f"${p:,.2f} (touched {n} times)" for p, n in levels)

    lines = [
        f"Latest price: ${price:,.2f} (most recent price on {last_day:%b %d}, from {source}).{stale}",
        f"Candles run through: {last_day:%b %d, %Y}",
        f"Trend: {trend}",
        f"50-day average: {money(s50)}",
        f"200-day average: {money(s200)}",
        f"Change over 1 day: {pct(change(1))}",
        f"Change over 1 month: {pct(change(21))}",
        f"Change over 3 months: {pct(change(63))}",
        f"52-week high: ${max(c['h'] for c in year):,.2f}",
        f"52-week low: ${min(c['l'] for c in year):,.2f}",
        f"Nearest support below price: {fmt(supports)}",
        f"Nearest resistance above price: {fmt(resistances)}",
        f"RSI (14-day): {num(r)}",
        f"Latest day's volume vs 20-day average: {last_vol / vol20:.2f}x (may be a partial day)",
    ]
    return {
        "text": "\n".join(lines),
        "daily": daily,
        "supports": supports,
        "resistances": resistances,
    }

def chart_html(ticker, chart):
    daily = chart["daily"]
    closes = [c["c"] for c in daily]
    ma50, ma200 = sma_series(closes, 50), sma_series(closes, 200)
    show = range(max(0, len(daily) - 250), len(daily))

    def day(c):
        return candle_date(c).strftime("%Y-%m-%d")

    candles = [{"time": day(daily[i]), "open": daily[i]["o"], "high": daily[i]["h"],
                "low": daily[i]["l"], "close": daily[i]["c"]} for i in show]
    volumes = [{"time": day(daily[i]), "value": daily[i]["v"],
                "color": "rgba(46,158,79,0.4)" if daily[i]["c"] >= daily[i]["o"] else "rgba(224,72,72,0.4)"}
               for i in show]
    line50 = [{"time": day(daily[i]), "value": ma50[i]} for i in show if ma50[i] is not None]
    line200 = [{"time": day(daily[i]), "value": ma200[i]} for i in show if ma200[i] is not None]
    levels = (
        [{"price": round(p, 2), "color": "#2e9e4f", "title": "Support"} for p, _ in chart["supports"]]
        + [{"price": round(p, 2), "color": "#e04848", "title": "Resistance"} for p, _ in chart["resistances"]]
    )
    box = f"chart_{ticker}_{len(daily)}"

    return f"""
    <div style='max-width:760px;font-family:sans-serif;margin-top:14px'>
      <div style='font-weight:700;margin-bottom:4px'>{html.escape(ticker)} daily candles, last 12 months</div>
      <div style='font-size:12px;opacity:.7;margin-bottom:6px'>
        Blue = 50-day average. Orange = 200-day average. Dashed green = support. Dashed red = resistance.
        Scroll to zoom, drag to move.
      </div>
      <div id='{box}' style='width:100%;height:420px;background:#ffffff'></div>
    </div>
    <script>
    (function() {{
      function draw() {{
        var el = document.getElementById('{box}');
        var chart = LightweightCharts.createChart(el, {{
          width: el.clientWidth || 740, height: 420,
          layout: {{ background: {{ color: '#ffffff' }}, textColor: '#222222' }},
          grid: {{ vertLines: {{ color: '#eeeeee' }}, horzLines: {{ color: '#eeeeee' }} }},
          rightPriceScale: {{ borderVisible: false }},
          timeScale: {{ borderVisible: false }}
        }});
        var candles = chart.addCandlestickSeries({{
          upColor: '#2e9e4f', downColor: '#e04848', borderVisible: false,
          wickUpColor: '#2e9e4f', wickDownColor: '#e04848'
        }});
        candles.setData({json.dumps(candles)});
        var vol = chart.addHistogramSeries({{ priceFormat: {{ type: 'volume' }}, priceScaleId: '' }});
        vol.priceScale().applyOptions({{ scaleMargins: {{ top: 0.82, bottom: 0 }} }});
        vol.setData({json.dumps(volumes)});
        var a = chart.addLineSeries({{ color: '#2962ff', lineWidth: 1, priceLineVisible: false, lastValueVisible: false }});
        a.setData({json.dumps(line50)});
        var b = chart.addLineSeries({{ color: '#f59f00', lineWidth: 1, priceLineVisible: false, lastValueVisible: false }});
        b.setData({json.dumps(line200)});
        {json.dumps(levels)}.forEach(function(l) {{
          candles.createPriceLine({{ price: l.price, color: l.color, lineWidth: 1, lineStyle: 2,
                                     axisLabelVisible: true, title: l.title }});
        }});
        chart.timeScale().fitContent();
      }}
      if (window.LightweightCharts) {{ draw(); }}
      else {{
        var s = document.createElement('script');
        s.src = 'https://unpkg.com/lightweight-charts@4.2.0/dist/lightweight-charts.standalone.production.js';
        s.onload = draw;
        document.head.appendChild(s);
      }}
    }})();
    </script>
    """

print("Chart step ready.")

In [ ]:
import html
from IPython.display import HTML, display

HEADINGS = [
    "WHERE THE EVIDENCE LEANS", "THE DECISION", "BEST BULL POINTS", "BEST BEAR POINTS",
    "WHERE THEY DISAGREE", "WHAT THE CHART SHOWS", "DATA WARNINGS",
    "CLAIMS THAT FAILED", "CHECK THIS YOURSELF",
]
COLORS = {"BEST BULL POINTS": "#2e9e4f", "BEST BEAR POINTS": "#e04848", "THE DECISION": "#2962ff"}
SUB_LABELS = ["Evidence:", "Why it matters:", "Other side:"]

def brief_to_html(brief):
    # Each section is [title, points]. Each point is [text, [(label, text), ...]].
    sections, current = [], None
    for raw in brief.splitlines():
        line = raw.strip()
        if not line:
            continue
        if line.upper().rstrip(":") in HEADINGS:
            current = [line.upper().rstrip(":"), []]
            sections.append(current)
            continue
        if current is None:
            continue
        clean = line.lstrip("-• ").strip()
        label = next((s for s in SUB_LABELS if clean.lower().startswith(s.lower())), None)
        if label and current[1]:
            current[1][-1][1].append((label, clean[len(label):].strip()))
        else:
            current[1].append([clean, []])

    # If the judge did not follow the format, show the plain text instead.
    if not sections:
        return f"<pre style='white-space:pre-wrap'>{html.escape(brief)}</pre>"

    out = []
    for title, points in sections:
        if title == "WHERE THE EVIDENCE LEANS":
            lean = html.escape(" ".join(p[0] for p in points))
            out.append(f"<div style='font-size:17px;font-weight:700;margin:6px 0 14px 0'>{lean}</div>")
            continue

        color = COLORS.get(title, "inherit")
        items = []
        for text, subs in points:
            if title == "THE DECISION" and ":" in text:
                lab, rest = text.split(":", 1)
                main = f"<b>{html.escape(lab)}:</b>{html.escape(rest)}"
            else:
                weight = "600" if subs else "400"
                main = f"<span style='font-weight:{weight}'>{html.escape(text)}</span>"
            sub_html = "".join(
                f"<div style='font-size:13px;margin:2px 0 0 0;opacity:.9'>"
                f"<b>{html.escape(lab)}</b> {html.escape(val)}</div>"
                for lab, val in subs
            )
            items.append(f"<li style='margin:7px 0'>{main}{sub_html}</li>")

        box = "border-left:3px solid #2962ff;padding-left:12px;" if title == "THE DECISION" else ""
        out.append(
            f"<div style='margin-top:14px;{box}'>"
            f"<div style='font-weight:700;color:{color}'>{title}</div>"
            f"<ul style='margin:4px 0 0 0'>{''.join(items)}</ul></div>"
        )
    return "".join(out)

def analyze(ticker, horizon):
    label = "Short-term trade" if horizon == "short" else "Long-term hold"
    print(f"Working on {ticker} ({label}). This takes a minute or two.\n")

    facts, name, today = get_facts(ticker)
    print("1/5 Got the numbers")

    try:
        chart = get_chart(ticker, today)
        chart_text = chart["text"]
        print("2/5 Got the chart")
    except Exception as e:
        chart = None
        chart_text = "No chart data is available for this stock."
        print(f"2/5 Chart skipped ({e})")

    findings = research(ticker, name, today, horizon)
    print("3/5 Research done")

    data = (
        f"DATA:\n{facts}\n\n"
        f"PRICE CHART DATA (part of DATA):\n{chart_text}\n\n"
        f"{HOW_TO_READ}\n\n{CHART_GUIDE}\n\n"
        f"RESEARCH:\n{findings}"
    )
    d = run_debate(data, horizon)
    print("4/5 Debate done")

    brief = judge(data, d, horizon)
    print("5/5 Judge done\n")

    details = "\n\n".join([
        "REAL DATA\n" + facts,
        "PRICE CHART DATA\n" + chart_text,
        "RESEARCH (Grok)\n" + findings,
        f"BULL CASE ({d['bull_name']})\n" + d["bull"],
        f"BEAR CASE ({d['bear_name']})\n" + d["bear"],
        f"BULL REPLY ({d['bull_name']})\n" + d["bull_reply"],
        f"BEAR REPLY ({d['bear_name']})\n" + d["bear_reply"],
    ])

    page = f"""
    <div style='border:1px solid #8886;border-radius:10px;padding:16px 20px;max-width:760px;
                font-family:sans-serif;font-size:14px;line-height:1.5'>
      <div style='font-size:20px;font-weight:700'>{html.escape(str(name))} ({html.escape(ticker)})</div>
      <div style='opacity:.7;font-size:12px'>{today} · {label} · Bull = {d['bull_name']} · Bear = {d['bear_name']} · Judge = Grok</div>
      {brief_to_html(brief)}
      <div style='opacity:.7;margin-top:14px;font-size:12px'>Research, not financial advice.</div>
    </div>
    {chart_html(ticker, chart) if chart else ""}
    <details style='max-width:760px;margin-top:12px;font-family:sans-serif'>
      <summary style='cursor:pointer;font-weight:700'>Show full details (data, chart numbers, research, full debate)</summary>
      <pre style='white-space:pre-wrap;font-size:13px'>{html.escape(details)}</pre>
    </details>
    """
    display(HTML(page))

ticker = input("Ticker: ").strip().upper()
answer = input("Short-term trade or long-term hold? Type short or long: ").strip().lower()
analyze(ticker, "short" if answer.startswith("s") else "long")